# Sangria noise-only segment plus a JAX eccentric waveform injection

This notebook reads the LISA Data Challenge Sangria training HDF5 file from the Zenodo record https://zenodo.org/records/7132178 and builds an example data product containing:

- `noise`: instrumental noise inferred as `obs/tdi - sky/dgb/tdi - sky/igb/tdi - sky/vgb/tdi - sky/mbhb/tdi` from the same HDF5 file.
- `eccentric_signal`: an eccentric binary generated by this repository's brute-force JAX eccentric waveform and link response path, then passed through pyTDI.
- `injected`: `noise + eccentric_signal`.

This notebook intentionally uses only `egb_jax_eccentric.eccentric_xyz_jax`. It does not import or call the eGB-multi trilinear implementation.

Physics/convention note: the Sangria file used here stores dimensionless `X`, `Y`, `Z` Michelson TDI data with `TDI_GENERATION = 1.5`. The local pyTDI bridge in this package can evaluate generation `1` or `2`, but pyTDI rejects `1.5`. Therefore, the default below is an explicitly marked demonstration injection, not a same-convention Sangria validation product. Set `ALLOW_APPROXIMATE_TDI_INJECTION = False` if you want the notebook to stop whenever that convention mismatch is present.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import platform
import sys
import warnings

import h5py
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import welch

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src" / "egb_jax_eccentric").exists():
            return candidate
    raise RuntimeError("Could not find repo root containing src/egb_jax_eccentric")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
sys.path.insert(0, str(REPO_ROOT / "src"))

from egb_jax_eccentric import EccentricBinaryParams, default_lisaorbits, eccentric_xyz_jax, state_from_lisaorbits
from egb_jax_eccentric.constants import KILOPARSEC_M

XYZ = ("X", "Y", "Z")
print("repo root:", REPO_ROOT)
print("python:", platform.python_version())

## Configuration

The default segment is one day at Sangria's 5 s TDI cadence. Increase `SEGMENT_DURATION_DAYS` only after the small run works; the full year has 6,307,200 samples per channel.

In [ ]:
SANGRIA_PATH = Path.home() / "Downloads" / "LDC2_sangria_training_v2.h5"
OUTPUT_PATH = REPO_ROOT / "outputs" / "sangria_noise_plus_jax_eccentric_example.h5"

SEGMENT_START_S = 0.0
SEGMENT_DURATION_DAYS = 1.0

# Sangria reports TDI_GENERATION = 1.5. This package's pyTDI bridge supports 1 or 2.
ALLOW_APPROXIMATE_TDI_INJECTION = True
EGB_PYTDI_GENERATION = 1
MEASUREMENT_ORDER = 3
DELAY_ORDER = 3

# A deliberately visible compact-binary example. Distances this small are for a notebook-scale injection demo.
SOURCE_PARAMS = {
    "f0_hz": 2.0e-3,
    "eccentricity": 0.05,
    "m1_solar": 0.5,
    "m2_solar": 0.5,
    "distance_m": 0.1 * KILOPARSEC_M,
    "beta": 0.3,
    "lambda_": 1.1,
    "psi": 0.4,
    "inclination": 0.8,
    "phi0": 0.2,
    "fdot": 0.0,
}
INJECTION_SCALE = 1.0

if not SANGRIA_PATH.exists():
    raise FileNotFoundError(f"Sangria training file not found: {SANGRIA_PATH}")

print("input:", SANGRIA_PATH)
print("output:", OUTPUT_PATH)
print("source:", json.dumps(SOURCE_PARAMS, indent=2, default=float))

In [ ]:
def scalar(h5: h5py.File, key: str):
    value = h5[key][()]
    if isinstance(value, bytes):
        return value.decode()
    if hasattr(value, "item"):
        return value.item()
    return value


def segment_bounds(h5: h5py.File, start_s: float, duration_days: float) -> tuple[int, int, float, float]:
    dset = h5["obs/tdi"]
    dt = float(dset.attrs["dt"])
    t0 = float(dset.attrs["t0"])
    if start_s < t0:
        raise ValueError("SEGMENT_START_S is before the dataset t0")
    start = int(round((start_s - t0) / dt))
    count = int(round(duration_days * 86400.0 / dt))
    stop = min(start + count, dset.shape[0])
    if stop <= start:
        raise ValueError("empty segment requested")
    return start, stop, dt, t0


def read_tdi_segment(h5: h5py.File, path: str, start: int, stop: int) -> dict[str, np.ndarray]:
    dset = h5[path]
    names = dset.dtype.names
    if dset.ndim != 2 or dset.shape[1] != 1 or names is None or not set(("t", *XYZ)).issubset(names):
        raise TypeError(f"{path} is not the expected structured TDI dataset")
    block = dset[start:stop, 0]
    return {name: np.asarray(block[name], dtype=np.float64) for name in ("t", *XYZ)}


def assert_same_grid(reference: dict[str, np.ndarray], candidate: dict[str, np.ndarray], path: str) -> None:
    if candidate["t"].shape != reference["t"].shape:
        raise ValueError(f"{path} has incompatible time shape {candidate['t'].shape}")
    if not np.array_equal(candidate["t"], reference["t"]):
        raise ValueError(f"{path} is not on the same timestamp grid")


def load_noise_only_training_segment(h5: h5py.File, start: int, stop: int):
    source_paths = ["sky/dgb/tdi", "sky/igb/tdi", "sky/vgb/tdi", "sky/mbhb/tdi"]
    missing = [path for path in source_paths if path not in h5]
    if missing:
        raise RuntimeError(
            "Cannot construct noise-only data from this file. Missing same-file source TDI components: "
            + ", ".join(missing)
        )

    obs = read_tdi_segment(h5, "obs/tdi", start, stop)
    noise = {"t": obs["t"].copy(), **{channel: obs[channel].copy() for channel in XYZ}}
    components = {}
    for path in source_paths:
        comp = read_tdi_segment(h5, path, start, stop)
        assert_same_grid(obs, comp, path)
        components[path] = comp
        for channel in XYZ:
            noise[channel] -= comp[channel]
    return noise, obs, components


with h5py.File(SANGRIA_PATH, "r") as h5:
    start, stop, dt, t0 = segment_bounds(h5, SEGMENT_START_S, SEGMENT_DURATION_DAYS)
    sangria_metadata = {
        "obs_tdi_dt_s": float(h5["obs/tdi"].attrs["dt"]),
        "obs_tdi_t0_s": float(h5["obs/tdi"].attrs["t0"]),
        "obs_tdi_units": str(h5["obs/tdi"].attrs.get("units", "")),
        "obs_orbit_type": scalar(h5, "obs/config/orbit_type") if "obs/config/orbit_type" in h5 else "unknown",
        "nominal_arm_length_m": scalar(h5, "obs/config/nominal_arm_length") if "obs/config/nominal_arm_length" in h5 else None,
        "tdi_generation": scalar(h5, "instru/config/TDI_GENERATION") if "instru/config/TDI_GENERATION" in h5 else None,
        "tdi_input_format": scalar(h5, "instru/config/TDI_INPUT_FORMAT") if "instru/config/TDI_INPUT_FORMAT" in h5 else None,
    }
    noise_xyz, obs_xyz, source_components = load_noise_only_training_segment(h5, start, stop)

print("segment samples:", stop - start)
print("segment start/stop seconds:", noise_xyz["t"][0], noise_xyz["t"][-1])
print("Sangria metadata:", json.dumps(sangria_metadata, indent=2, default=float))
print("noise max abs:", {channel: float(np.max(np.abs(noise_xyz[channel]))) for channel in XYZ})

In [ ]:
sangria_tdi_generation = sangria_metadata["tdi_generation"]
if sangria_tdi_generation is not None and float(sangria_tdi_generation) != float(EGB_PYTDI_GENERATION):
    message = (
        f"Sangria stores TDI_GENERATION={sangria_tdi_generation}, but this signal uses "
        f"egb_jax_eccentric.eccentric_xyz_jax with pyTDI generation={EGB_PYTDI_GENERATION}. "
        "This is an approximate demonstration unless this package is extended to reproduce the Sangria 1.5 convention."
    )
    if not ALLOW_APPROXIMATE_TDI_INJECTION:
        raise RuntimeError(message)
    warnings.warn(message)

t = noise_xyz["t"]
state = state_from_lisaorbits(default_lisaorbits("equal"), t)
source = EccentricBinaryParams(
    mean_motion=np.pi * SOURCE_PARAMS["f0_hz"],
    eccentricity=SOURCE_PARAMS["eccentricity"],
    m1_solar=SOURCE_PARAMS["m1_solar"],
    m2_solar=SOURCE_PARAMS["m2_solar"],
    distance_m=SOURCE_PARAMS["distance_m"],
    beta=SOURCE_PARAMS["beta"],
    lambda_=SOURCE_PARAMS["lambda_"],
    psi=SOURCE_PARAMS["psi"],
    inclination=SOURCE_PARAMS["inclination"],
    phi0=SOURCE_PARAMS["phi0"],
    fdot=SOURCE_PARAMS["fdot"],
)

raw_signal_xyz = eccentric_xyz_jax(
    state,
    source,
    batch_size=1,
    physics_mode="1pn_periastron",
    generation=EGB_PYTDI_GENERATION,
    measurement_order=MEASUREMENT_ORDER,
    delay_order=DELAY_ORDER,
)
signal_backend = "egb_jax_eccentric.eccentric_xyz_jax"

# The local response is an analytic complex representation. Sangria stores real dimensionless TDI series.
signal_xyz = {channel: np.real(np.asarray(raw_signal_xyz[channel], dtype=np.complex128)).astype(np.float64) for channel in XYZ}
imaginary_fraction = {}
for channel in XYZ:
    arr = np.asarray(raw_signal_xyz[channel], dtype=np.complex128)
    denom = max(float(np.max(np.abs(np.real(arr)))), np.finfo(float).tiny)
    imaginary_fraction[channel] = float(np.max(np.abs(np.imag(arr))) / denom)

injected_xyz = {"t": t.copy(), **{channel: noise_xyz[channel] + INJECTION_SCALE * signal_xyz[channel] for channel in XYZ}}

print("signal backend:", signal_backend)
print("signal max abs:", {channel: float(np.max(np.abs(signal_xyz[channel]))) for channel in XYZ})
print("analytic-signal imaginary/real max fractions:", imaginary_fraction)

In [ ]:
plot_count = min(t.size, int(6 * 3600 / dt))
fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
for ax, channel in zip(axes, XYZ):
    tt_hours = (t[:plot_count] - t[0]) / 3600.0
    ax.plot(tt_hours, noise_xyz[channel][:plot_count], lw=0.8, label="noise only")
    ax.plot(tt_hours, injected_xyz[channel][:plot_count], lw=0.8, label="noise + eccentric")
    ax.set_ylabel(f"{channel}")
    ax.grid(alpha=0.25)
axes[-1].set_xlabel("time since segment start [h]")
axes[0].legend(loc="upper right")
fig.suptitle("Sangria-derived noise-only segment with JAX eccentric injection")
fig.tight_layout()
plt.show()

fs = 1.0 / dt
nperseg = min(8192, max(256, 2 ** int(np.floor(np.log2(t.size // 4)))))
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharey=True)
for ax, channel in zip(axes, XYZ):
    freq, p_noise = welch(noise_xyz[channel], fs=fs, nperseg=nperseg, detrend="constant")
    _, p_signal = welch(signal_xyz[channel], fs=fs, nperseg=nperseg, detrend="constant")
    mask = freq > 0
    ax.loglog(freq[mask], np.sqrt(p_noise[mask]), label="noise ASD")
    ax.loglog(freq[mask], np.sqrt(p_signal[mask]), label="signal ASD")
    ax.axvline(SOURCE_PARAMS["f0_hz"], color="k", ls=":", lw=1.0)
    ax.set_title(channel)
    ax.set_xlabel("frequency [Hz]")
    ax.grid(alpha=0.25, which="both")
axes[0].set_ylabel("ASD [1/sqrt(Hz)]")
axes[0].legend(loc="best")
fig.tight_layout()
plt.show()

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with h5py.File(OUTPUT_PATH, "w") as out:
    out.attrs["source_record"] = "Zenodo 10.5281/zenodo.7132178, LISA Data Challenge Sangria (LDC2a)"
    out.attrs["input_hdf5"] = str(SANGRIA_PATH)
    out.attrs["segment_start_s"] = float(t[0])
    out.attrs["segment_stop_s"] = float(t[-1])
    out.attrs["segment_dt_s"] = float(dt)
    out.attrs["noise_definition"] = "obs/tdi minus sky/dgb, sky/igb, sky/vgb, and sky/mbhb TDI components from the same Sangria training file"
    out.attrs["sangria_metadata_json"] = json.dumps(sangria_metadata, default=float)
    out.attrs["egb_source_params_json"] = json.dumps(SOURCE_PARAMS, default=float)
    out.attrs["egb_signal_backend"] = signal_backend
    out.attrs["egb_pytdi_generation"] = float(EGB_PYTDI_GENERATION)
    out.attrs["injection_scale"] = float(INJECTION_SCALE)
    out.attrs["convention_status"] = "approximate demonstration: Sangria stores TDI_GENERATION=1.5 while this egb_jax_eccentric signal uses pyTDI generation 1 or 2"

    out.create_dataset("t", data=t, compression="gzip")
    for group_name, series in (("noise", noise_xyz), ("eccentric_signal", signal_xyz), ("injected", injected_xyz)):
        group = out.create_group(group_name)
        for channel in XYZ:
            group.create_dataset(channel, data=series[channel], compression="gzip")

print("wrote", OUTPUT_PATH)
with h5py.File(OUTPUT_PATH, "r") as check:
    print("groups:", list(check.keys()))
    print("samples:", check["t"].shape[0])
    print("max |injected - noise|:", {channel: float(np.max(np.abs(check["injected"][channel][:] - check["noise"][channel][:]))) for channel in XYZ})

In [ ]:
FIGURE_DIR = REPO_ROOT / "notebooks" / "sangria_noise_eccentric_injection_plots"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

plot_labels = {
    "noise": "Sangria-derived instrumental noise",
    "eccentric_signal": "JAX eccentric binary signal",
    "injected": "Noise plus JAX eccentric binary signal",
}
plot_files = {
    "noise": "sangria_noise_only_xyz.png",
    "eccentric_signal": "jax_eccentric_signal_xyz.png",
    "injected": "sangria_noise_plus_jax_eccentric_xyz.png",
}
plot_colors = {"X": "#1f77b4", "Y": "#d62728", "Z": "#2ca02c"}
plot_count = min(t.size, int(6 * 3600 / dt))
tt_hours = (t[:plot_count] - t[0]) / 3600.0
series_by_group = {"noise": noise_xyz, "eccentric_signal": signal_xyz, "injected": injected_xyz}

for group_name, series in series_by_group.items():
    fig, axes = plt.subplots(3, 1, figsize=(10.5, 7.0), sharex=True)
    for ax, channel in zip(axes, XYZ):
        y = series[channel][:plot_count]
        ax.plot(tt_hours, y, lw=0.75, color=plot_colors[channel])
        ax.set_ylabel(f"{channel} TDI")
        ax.grid(alpha=0.28)
        finite = np.isfinite(y)
        if finite.any():
            span = max(float(np.max(y[finite]) - np.min(y[finite])), np.finfo(float).tiny)
            pad = 0.08 * span
            ax.set_ylim(float(np.min(y[finite]) - pad), float(np.max(y[finite]) + pad))
    axes[-1].set_xlabel("time since segment start [h]")
    fig.suptitle(plot_labels[group_name])
    fig.text(
        0.01,
        0.01,
        "Sangria training noise = obs/tdi - same-file sky TDI components; signal via egb_jax_eccentric.eccentric_xyz_jax, pyTDI generation 1.",
        fontsize=8,
        color="#444444",
    )
    fig.tight_layout(rect=(0, 0.035, 1, 0.96))
    output_figure = FIGURE_DIR / plot_files[group_name]
    fig.savefig(output_figure, dpi=180)
    plt.close(fig)
    print("wrote", output_figure)